In [1]:
import numpy as np
from nanover.recording import NanoverRecordingReader
from nanover.utilities.state_dictionary import DictionaryChange
from nanover.imd.imd_state import dict_to_interaction
from follower import Checkpoint, Pin


IN_PATH = "knot-tying-checkpoints-2.nanover.zip"
RESTRAINT_PREFIX = "interaction.MOVEABLE-RESTRAINT"
CHECKPOINT_KEY = "mark.checkpoint"


CHECKPOINTS: list[Checkpoint] = []


with NanoverRecordingReader.from_path(IN_PATH) as reader:
    index: dict[str, set[int]] = {}

    for event in reader.iter_max():
        if event.next_state_event is None:
            continue

        change = DictionaryChange.from_dict(event.next_state_event.message)

        if CHECKPOINT_KEY not in change.updates:
            continue

        checkpoint = Checkpoint()
        positions = event.next_frame.particle_positions

        for key in event.next_state:
            if key.startswith(RESTRAINT_PREFIX):
                restraint = dict_to_interaction(event.next_state[key])
                particles = [int(i) for i in restraint.particles]
                centroid = np.average(positions[particles], axis=0)

                pin = Pin(particles=particles, position=centroid)
                checkpoint.pins.append(pin)

        if checkpoint.pins:
            CHECKPOINTS.append(checkpoint)

CHECKPOINTS

[Checkpoint(pins=[Pin(particles=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], position=array([2.3507342, 2.239971 , 4.6463   ], dtype=float32)), Pin(particles=[162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172], position=array([7.663963 , 2.5156038, 4.213367 ], dtype=float32))]),
 Checkpoint(pins=[Pin(particles=[162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172], position=array([7.6801634, 2.4426768, 4.237811 ], dtype=float32)), Pin(particles=[82, 83, 84, 85, 86, 87, 88, 89, 90, 91], position=array([4.84034  , 2.2713134, 3.5504315], dtype=float32)), Pin(particles=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], position=array([4.742126  , 3.3494623 , 0.63086003], dtype=float32))]),
 Checkpoint(pins=[Pin(particles=[162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172], position=array([7.6868563, 2.3530765, 4.1614566], dtype=float32)), Pin(particles=[82, 83, 84, 85, 86, 87, 88, 89, 90, 91], position=array([4.822738 , 2.3397284, 3.5758877], dtype=float32)), Pin(particles=[42, 43, 44, 45, 46, 47

In [2]:
## Setup runner
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="checkpoint replay")
imd_runner.load(0)

In [3]:
from nanover.websocket import NanoverImdClient

client = NanoverImdClient.from_runner(imd_runner)
with client.root_selection.modify() as selection:
    selection.renderer = "cartoon"

In [4]:
def notify_all(message):
    for command in imd_runner.app_server.commands:
        if "/notify" in command:
            imd_runner.app_server.run_command(command, dict(message=message))

In [5]:
from follower import Follower, Group, Path

CHECKPOINT_INDEX = 0
FOLLOWER: Follower | None = None


def get_current_centroid(particles: list[int]):
    frame = imd_runner.app_server.frame_publisher.current_frame
    centroid = np.average(frame.particle_positions[particles], axis=0)
    return centroid


def replay_start():
    global FOLLOWER, CHECKPOINT_INDEX

    notify_all(f"REPLAYING CHECKPOINT {CHECKPOINT_INDEX}")

    checkpoint = CHECKPOINTS[CHECKPOINT_INDEX]
    CHECKPOINT_INDEX += 1

    groups = [
        Group(particles=pin.particles, path=Path.from_points([get_current_centroid(pin.particles), pin.position]))
        for pin in checkpoint.pins
    ]

    if FOLLOWER is not None:
        FOLLOWER.stop()

    FOLLOWER = Follower.from_runner(imd_runner)
    FOLLOWER.start(groups, release=False, speed=0.01)


def replay_cancel():
    global FOLLOWER, CHECKPOINT_INDEX
    FOLLOWER.stop()
    FOLLOWER = None
    CHECKPOINT_INDEX = 0

    notify_all(f"RESETING REPLAY")


imd_runner.app_server.register_command("user/replay/start", replay_start)
imd_runner.app_server.register_command("user/replay/cancel", replay_cancel)

In [6]:
from ipywidgets import Output
output = Output()
output

Output()

In [ ]:
with output:
    print("test")